# Energy Model — Helena's Exploration

## Step 1: Explore the Trassenfinder API
Goal: understand the response structure before collecting real data.

In [3]:
import requests

response = requests.get("https://openapi.trassenfinder.de/")
print(response.status_code)

200


**Result:** Status code 200 - API is reachable.
next: inspect available endpoints via the Swagger docs page.


## Step 1 findings — first manual test route

Test route: [your start] → [your destination]

Results returned by Trassenfinder:
- Distance: 644.1 km
- Travel time: 10:36 h
- Energy consumption: 10,101 kWh
- Cost: 2,534 €
- Additional detail available via "Fahrtverlaufsdiagramm" (detailed graph)

Two route options are offered: "Gewichtet" (weighted) vs "Kürzeste" (shortest).
Weighting between distance/time/energy is adjustable via sliders ("Gewichtung").

## Step 1 findings — API request structure

Endpoint: POST https://trassenfinder.de/api/web/routen/suche
Content-Type: application/json

Key request fields identified:
- wagenzugmasse_t → train weight (tonnes) ✅ matches our `total_weight_t` input
- streckenklasse → track class, possible proxy for terrain
- wegpunkte → start/end stations (DS100 codes)
- gewichtung_parameter → weighting between distance/time/energy

Still need: response structure (checking Preview tab next)

## Step 1b: Replicate the request in Python

In [4]:
import requests
import json

url = "https://trassenfinder.de/api/web/routen/suche"

payload = {
    "infrastruktur_id": 8,
    "sucheinstellungen": {
        "verkehrsart": "sgv",
        "an_abzeit": "2026-07-28T17:50:04+02:00",
        "zeitvorgabe_typ": "abzeit",
        "optimierungsvarianten_berechnen": True,
        "richtungswechsel_zulaessig": True,
        "rangierfahrt_zulaessig": False,
        "vermeidung_parameter": {
            "ueberlastete_meiden": False, "sbahnen_meiden": True,
            "nebenbahnen_meiden": False, "schnellfahrstrecken_meiden": True,
            "knotenbahnhoefe_meiden": True, "notbremsueberbrueckung_meiden": False,
            "wirbelstrombremse_meiden": False, "eingleisige_strecken_meiden": False,
            "strecken_mit_vorrang_sgv_meiden": False, "strecken_mit_vorrang_spv_meiden": False
        },
        "initiale_sperrungen_beruecksichtigen": True,
        "wendezeit_min": 30,
        "mit_realistischen_fahrzeiten_optimieren": True,
        "verkehrshalte_nur_an_bahnsteigen": False,
        "einschraenkungen_beachten": True,
        "manueller_fahrzeitzuschlag_prozent": 0,
        "einzelgrenzlastberechnung_zulaessig": False,
        "bauzuschlaege_beachten": False,
        "laengenabhaengige_grenzlasten_verwenden": False,
        "tpn_triebfahrzeugbezeichnung_anzeigen": False,
        "gewichtung_parameter": {"streckenlaenge_prozent": 40, "fahrzeit_prozent": 30, "energie_prozent": 30},
        "zusatzkosten_parameter": {
            "energiebezugspreis_euro_pro_kwh": 0.18, "rueckspeisung_euro_pro_kwh": 0.09,
            "kosten_besetzte_tfz_inkl_personal_euro_pro_h": 150, "kosten_unbesetzte_tfz_euro_pro_h": 80,
            "kosten_wagenzug_euro_pro_h": 150, "kostenpauschale_ungekuppelt_nachschieben_euro": 999,
            "zusaetzlicher_energieverbrauch_pro_wagen_kw": 0,
            "energieverbrauch_hilfsbetriebe_und_wagen_beachten": True
        },
        "trassenentgelt_parameter": {"marktsegment": "sgv_standard", "flexibilitaet": "kein_flex", "prioritaet": "keine_prioritaet", "gefahrgutganzzug": False}
    },
    "wegpunkte": [
        {
            "zugcharakteristik": {
                "bremshundertstel": 70, "bremsstellung": "P", "aktive_neigetechnik": False,
                "kupplungsbauart": "kn450", "dla_u_profile": [], "fuehrendes_fahrzeug": "lokomotive",
                "kv_profil": {"p": "N", "c": "N"}, "nachschiebeart": "ohne", "streckenklasse": "D4",
                "traktionsartwechsel": False,
                "triebfahrzeug": {"hauptnummer": "6185", "unternummer": 2, "kennung": "L", "kennung_wert": 80},
                "vorspannart": "ohne", "wagenzuglaenge_m": 600, "wagenzugmasse_t": 1200,
                "wagenanzahl": 0, "v_max": 100,
                "zugbeeinflussung_parameter": {"etcs_system_version": "ohne", "lzb": True, "pzb": True}
            },
            "betriebsstelle": {"ds100": "HH", "mutter": True}
        },
        {"betriebsstelle": {"ds100": "MH", "mutter": True}}
    ],
    "nutzer_sperrungen": []
}

headers = {
    "Content-Type": "application/json",
    "Accept": "application/json",
    "Origin": "https://trassenfinder.de",
    "Referer": "https://trassenfinder.de/",
}

response = requests.post(url, json=payload, headers=headers)
print(response.status_code)
print(list(response.json().keys()))  # just show the top-level field names, not everything

200
['result']


## Step 1c: First successful API call from Python

Replicated the browser's request in Python using `requests.post()`.

Request: POST to https://trassenfinder.de/api/web/routen/suche
- Test route: HH (Hamburg) → MH (München)
- Train weight used: 1200 t (wagenzugmasse_t)
- No authentication/session cookie needed — request succeeded without it

Result: Status 200, response JSON has one top-level key: `result`

Next: inspect the structure of `result` to find the actual output fields 
(distance, energy, cost, weight limits) so we can extract them into a clean row.

In [5]:
data = response.json()["result"]
print(type(data))

if isinstance(data, dict):
    print(data.keys())
elif isinstance(data, list):
    print(len(data))
    print(data[0].keys() if isinstance(data[0], dict) else data[0])

<class 'dict'>
dict_keys(['id', 'gewichtete_route', 'kuerzeste_route', 'einschraenkungen_in_fahrzeitraum'])


## Step 1d: Inspect route result structure

Top-level result has two route options (matching UI): 
`gewichtete_route` (weighted) and `kuerzeste_route` (shortest),
plus `id` and `einschraenkungen_in_fahrzeitraum` (restrictions).

Next: look inside `gewichtete_route` for the actual output fields.

In [6]:
route = data["gewichtete_route"]
print(type(route))
print(route.keys() if isinstance(route, dict) else route)

<class 'dict'>
dict_keys(['routen_typ', 'routenpunkte', 'technische_abfahrt', 'realistische_abfahrt', 'zusammenfassung', 'maximalwerte', 'punkt_zu_punkt_geschwindigkeit_unzulaessig', 'boundingbox'])


## Step 1e: Found likely summary fields

`gewichtete_route` contains a `zusammenfassung` (summary) key — 
likely holds distance, time, energy, cost matching the UI display.

In [7]:
summary = route["zusammenfassung"]
print(summary)

{'fahrzeit_technisch_min': 425, 'fahrzeit_realistisch_min': 636, 'weglaenge_hm': 6441, 'energieverbrauch_kwh': 10101, 'marktsegment': 'sgv_standard', 'trassenpreis_euro': 2534, 'preis_energie_euro': 1853, 'kosten_fahrzeuge_personal_euro': 3176, 'nachschiebekosten_euro': 0}


## Step 1f: Confirmed field mapping (Trassenfinder → our model)

`zusammenfassung` fields decoded:
- energieverbrauch_kwh → energy_kwh (direct match, no conversion)
- weglaenge_hm → distance_km = weglaenge_hm / 10  (hm = hectometers = 100m units)
- fahrzeit_realistisch_min → travel time in minutes (÷60 for hours)
- trassenpreis_euro → track usage price (not needed for energy model)

Train weight (input, not output): wagenzugmasse_t, set in the request payload.

Step 1 complete: API confirmed working, request/response structure understood.
Next: Step 2 — write a function to run many routes and collect samples into a CSV.

## Step 2: Create a reusable API function

The API request works for a single route.

To collect many training samples, the request process is converted into a reusable function. The function will take a route and train weight as inputs and return the variables needed for the energy model.

In [16]:
def get_route_energy(route, weight_t):
    """
    Extract relevant variables for the energy model.
    """

    summary = route["zusammenfassung"]

    result = {
        "energy_kwh": summary["energieverbrauch_kwh"],
        "distance_km": summary["weglaenge_hm"] / 10,
        "travel_time_min": summary["fahrzeit_realistisch_min"],
        "weight_t": weight_t
    }

    return result

In [9]:
sample = get_route_energy(route)

print(sample)

{'energy_kwh': 10101, 'distance_km': 644.1, 'travel_time_min': 636}


#The extraction function was tested successfully and returns the variables required for the energy model.

In [11]:
samples = []

sample = get_route_energy(route)

samples.append(sample)

print(samples)

[{'energy_kwh': 10101, 'distance_km': 644.1, 'travel_time_min': 636}]


#The extracted route data is stored in a list. This structure will later be converted into a CSV dataset.

## Step 2a: Create automated route queries

#The API response structure is understood, so the next step is to automate the request process for multiple routes.

In [12]:
def extract_route_data(response):
    """
    Extract the weighted route from the API response.
    """

    data = response.json()["result"]

    route = data["gewichtete_route"]

    return route

#The API response can now be converted into a route object using a reusable function.

#let´s test, if the function creates the same result as before:

In [13]:
route = extract_route_data(response)

print(type(route))
print(route.keys())

<class 'dict'>
dict_keys(['routen_typ', 'routenpunkte', 'technische_abfahrt', 'realistische_abfahrt', 'zusammenfassung', 'maximalwerte', 'punkt_zu_punkt_geschwindigkeit_unzulaessig', 'boundingbox'])


#Explanation: Brick 1: "Extract_oute_data" - takes the entire API answer and finds the actual route ---- "route" ; Brick 2: "Get_route_energy" - takes only Variables from the route information that we need. WHY? to be flexible if something changes later in the API structure or we want a different variable - just change one brick.

## Step 2b: Combine API extraction and data extraction

#The individual processing steps are combined into one workflow that returns a complete training sample for one route.

In [17]:
def collect_single_sample(response, weight_t):
    """
    Convert one API response into one energy model sample.
    """

    route = extract_route_data(response)

    sample = get_route_energy(route, weight_t)

    return sample

#The complete processing chain now returns only the variables required for the energy model.

#test:

In [15]:
sample = collect_single_sample(response)

print(sample)

{'energy_kwh': 10101, 'distance_km': 644.1, 'travel_time_min': 636}


## Step 2c: Add train weight information

The train weight is added to each sample because it is an important predictor for energy consumption.

#test after changing some rows:

In [18]:
sample = collect_single_sample(response, 650)

print(sample)

{'energy_kwh': 10101, 'distance_km': 644.1, 'travel_time_min': 636, 'weight_t': 650}


## Step 2d: Collect multiple route samples

#Multiple route results will be stored in a list to create a training dataset for the energy model.

In [19]:
samples = []

print(samples)

[]


In [20]:
sample = collect_single_sample(response, 650)

samples.append(sample)

print(samples)

[{'energy_kwh': 10101, 'distance_km': 644.1, 'travel_time_min': 636, 'weight_t': 650}]


The first route sample has been added to the training dataset.

## Step 2e: Create a reusable API request function

#The API request is converted into a function so that different routes and train weights can be queried automatically.

In [21]:
def query_trassenfinder(start_ds100, end_ds100, weight_t):
    """
    Query Trassenfinder for one route.
    """

    payload["wegpunkte"][0]["betriebsstelle"]["ds100"] = start_ds100
    payload["wegpunkte"][1]["betriebsstelle"]["ds100"] = end_ds100
    payload["wegpunkte"][0]["zugcharakteristik"]["wagenzugmasse_t"] = weight_t

    response = requests.post(url, json=payload, headers=headers)

    return response

#Let´s test! 

In [22]:
test_response = query_trassenfinder("HH", "MH", 1200)

print(test_response.status_code)

200


## Step 2f: Create complete sample collection function

#The individual API and extraction steps are combined into one function that returns one complete training sample.

In [23]:
def collect_route_sample(start_ds100, end_ds100, weight_t):
    """
    Query one route and return the variables needed for the energy model.
    """

    response = query_trassenfinder(start_ds100, end_ds100, weight_t)

    route = extract_route_data(response)

    sample = get_route_energy(route, weight_t)

    return sample

Let´s test!

In [24]:
sample = collect_route_sample("HH", "MH", 1200)

print(sample)

{'energy_kwh': 10101, 'distance_km': 644.1, 'travel_time_min': 636, 'weight_t': 1200}


#The complete workflow can now generate one training sample from a route definition.

## Step 2g: Create a route list

#A list of different routes and train weights is created so that multiple training samples can be collected automatically.

In [25]:
routes = [
    {
        "start": "HH",
        "end": "MH",
        "weight_t": 1200
    },
    {
        "start": "HH",
        "end": "B",
        "weight_t": 1000
    },
    {
        "start": "B",
        "end": "FF",
        "weight_t": 900
    }
]

print(routes)

[{'start': 'HH', 'end': 'MH', 'weight_t': 1200}, {'start': 'HH', 'end': 'B', 'weight_t': 1000}, {'start': 'B', 'end': 'FF', 'weight_t': 900}]


In [26]:
samples = []

for route_info in routes:
    sample = collect_route_sample(
        route_info["start"],
        route_info["end"],
        route_info["weight_t"]
    )

    samples.append(sample)

print(samples)

KeyError: 'result'

In [2]:
import pandas as pd

ModuleNotFoundError: No module named 'pandas'

In [4]:
import pandas as pd

# Step 3: Load Complete DB Station Database

In this step, we load a nationwide Deutsche Bahn station dataset.

The previous dataset contained only a regional subset and was therefore not suitable for generating diverse training routes.

The goal of this step is:
- load the complete station database
- inspect available variables
- check if DS100 station codes and geographic information are available

This database will later be used to generate diverse routes for the energy model.

In [14]:
stations_full = pd.read_csv(
    file_path,
    sep=";"
)

stations_full.head()

,EVA_NR,DS100,NAME,VERKEHR,LAENGE,BREITE,Unnamed: 6,Unnamed: 7
0,8000001,KA,Aachen Hbf,FV,6.091499,50.767800,NaN,NaN
1,8070704,KASZ,Aachen Schanz,RV,6.073840,50.769862,NaN,NaN
2,8000404,KAW,Aachen West,RV,6.070715,50.780360,NaN,NaN
3,8000406,KAREP,Aachen-Rothe Erde,RV,6.116475,50.770202,NaN,NaN
4,8000002,TA,Aalen,FV,10.096271,48.841013,NaN,NaN


In [13]:
# Check first lines of the CSV file

with open(file_path, "r", encoding="utf-8") as f:
    for i in range(5):
        print(f.readline())

﻿EVA_NR;DS100;NAME;VERKEHR;LAENGE;BREITE;;

8000001;KA;Aachen Hbf;FV;6.091499;50.7678;;

8070704;KASZ;Aachen Schanz;RV;6.07384;50.769862;;

8000404;KAW;Aachen West;RV;6.070715;50.78036;;

8000406;KAREP;Aachen-Rothe Erde;RV;6.116475;50.770202;;



# Step 3.3: Inspect Station Database Size

In this step, we check the total number of stations and the available traffic categories.

This helps us understand the diversity of the dataset before creating training routes.

In [15]:
# Number of stations and columns

stations_full.shape

(6598, 8)

In [16]:
# Traffic categories

stations_full["VERKEHR"].value_counts()

VERKEHR
RV         4274
nur DPN    1998
FV          326
Name: count, dtype: int64

# Step 3.4: First Station Filtering

The complete database contains different types of railway stations.

For the energy model, we want to avoid very small local stops and focus on stations that represent meaningful railway routes.

In this step, we inspect the effect of filtering by traffic type.

In [17]:
# Create first filtered station dataset

stations_filtered = stations_full[
    stations_full["VERKEHR"].isin(["FV", "RV"])
]

stations_filtered.shape

(4600, 8)

# Step 3.5: Inspect Geographic Information

The station database contains geographic coordinates.

These coordinates will later allow us to create diverse routes covering different regions and terrain types in Germany.

In [18]:
# Check coordinate data

stations_filtered[["NAME", "LAENGE", "BREITE"]].head()

,NAME,LAENGE,BREITE
0,Aachen Hbf,6.091499,50.767800
1,Aachen Schanz,6.073840,50.769862
2,Aachen West,6.070715,50.780360
3,Aachen-Rothe Erde,6.116475,50.770202
4,Aalen,10.096271,48.841013


In [19]:
stations_filtered[["LAENGE", "BREITE"]].describe()

,LAENGE,BREITE
count,4600.000000,4600.000000
mean,10.028090,50.450256
std,2.128530,1.730738
min,6.070715,47.411032
25%,8.397927,49.105494
50%,9.742067,50.340741
75%,11.755921,51.538090
max,14.979080,54.906839


# Step 3.6: Inspect Traffic Distribution

Before creating the training routes, we inspect how many stations belong to each traffic category.

This helps us decide which stations should be included in the route generation process.

In [20]:
stations_filtered["VERKEHR"].value_counts()

VERKEHR
RV    4274
FV     326
Name: count, dtype: int64

In [21]:
stations_filtered[stations_filtered["VERKEHR"]=="FV"].head(20)

,EVA_NR,DS100,NAME,VERKEHR,LAENGE,BREITE,Unnamed: 6,Unnamed: 7
0,8000001,KA,Aachen Hbf,FV,6.091499,50.767800,NaN,NaN
4,8000002,TA,Aalen,FV,10.096271,48.841013,NaN,NaN
46,8000459,MAIN,Ainring,FV,12.964501,47.815033,NaN,NaN
64,8000483,HALF,Alfeld(Leine),FV,9.817779,51.981417,NaN,NaN
98,8000004,HA,Altenbeken,FV,8.943319,51.766433,NaN,NaN
149,8000331,KAND,Andernach,FV,7.404839,50.434542,NaN,NaN
150,8010004,WA,Angermünde,FV,13.996361,53.015221,NaN,NaN
155,8011044,WAK,Anklam,FV,13.701827,53.856102,NaN,NaN
163,8000009,NAN,Ansbach,FV,10.578239,49.298032,NaN,NaN
170,8011051,UAP,Apolda,FV,11.526136,51.030940,NaN,NaN


# Step 3.7: Clean Station Dataset

In this step, we create a cleaned version of the station database.

The raw CSV contains additional empty columns from the export.
We keep only the variables needed for route generation and later analysis.

The cleaned dataset will become the basis for creating a diverse training route dataset.

In [22]:
# Select relevant station attributes

stations_clean = stations_filtered[
    [
        "DS100",
        "NAME",
        "VERKEHR",
        "LAENGE",
        "BREITE"
    ]
]

stations_clean.head()

,DS100,NAME,VERKEHR,LAENGE,BREITE
0,KA,Aachen Hbf,FV,6.091499,50.767800
1,KASZ,Aachen Schanz,RV,6.073840,50.769862
2,KAW,Aachen West,RV,6.070715,50.780360
3,KAREP,Aachen-Rothe Erde,RV,6.116475,50.770202
4,TA,Aalen,FV,10.096271,48.841013


stations_clean.shape

# Step 3.8: Remove Duplicate Stations

The database can contain multiple entries for the same operational station.

In this step, we remove duplicate DS100 entries to ensure that each station is represented only once.

The DS100 code is used as the unique identifier because it is required for route requests.

In [24]:
# Remove duplicate stations based on DS100 code

stations_clean = stations_clean.drop_duplicates(
    subset="DS100"
)

stations_clean.shape

(4600, 5)

In [25]:
stations_clean["DS100"].nunique()

4600

# Step 3.9: Create Station Type Category

The traffic classification is used to categorize stations.

This information will later support the creation of a diverse route dataset.

Categories:
- FV: long-distance railway stations
- RV: regional railway stations

In [26]:
# Create station type category

stations_clean["station_type"] = stations_clean["VERKEHR"]

stations_clean.head()

,DS100,NAME,VERKEHR,LAENGE,BREITE,station_type
0,KA,Aachen Hbf,FV,6.091499,50.767800,FV
1,KASZ,Aachen Schanz,RV,6.073840,50.769862,RV
2,KAW,Aachen West,RV,6.070715,50.780360,RV
3,KAREP,Aachen-Rothe Erde,RV,6.116475,50.770202,RV
4,TA,Aalen,FV,10.096271,48.841013,FV


In [27]:
stations_clean["station_type"].value_counts()

station_type
RV    4274
FV     326
Name: count, dtype: int64

# Step 3.10: Create Geographic Regions

To generate a diverse training dataset, stations are grouped into broad geographic regions.

These regions are not used as exact administrative boundaries.
They are only used to ensure that later route generation covers different parts of Germany.

In [28]:
# Create rough geographic regions based on coordinates

def assign_region(row):
    lon = row["LAENGE"]
    lat = row["BREITE"]
    
    if lat >= 52:
        return "North"
    elif lat < 49:
        return "South"
    elif lon < 9:
        return "West"
    elif lon > 12:
        return "East"
    else:
        return "Central"


stations_clean["region"] = stations_clean.apply(assign_region, axis=1)

stations_clean.head()

,DS100,NAME,VERKEHR,LAENGE,BREITE,station_type,region
0,KA,Aachen Hbf,FV,6.091499,50.767800,FV,West
1,KASZ,Aachen Schanz,RV,6.073840,50.769862,RV,West
2,KAW,Aachen West,RV,6.070715,50.780360,RV,West
3,KAREP,Aachen-Rothe Erde,RV,6.116475,50.770202,RV,West
4,TA,Aalen,FV,10.096271,48.841013,FV,South


In [32]:
stations_clean["region"].value_counts()

region
West       1374
South      1062
North       896
Central     822
East        446
Name: count, dtype: int64

# Step 3.11: Final Station Dataset Check

The cleaned station database now contains:
- unique station identifiers
- traffic categories
- geographic information
- regional classification

Before exporting, we inspect the final structure.

In [33]:
stations_clean.head()

,DS100,NAME,VERKEHR,LAENGE,BREITE,station_type,region
0,KA,Aachen Hbf,FV,6.091499,50.767800,FV,West
1,KASZ,Aachen Schanz,RV,6.073840,50.769862,RV,West
2,KAW,Aachen West,RV,6.070715,50.780360,RV,West
3,KAREP,Aachen-Rothe Erde,RV,6.116475,50.770202,RV,West
4,TA,Aalen,FV,10.096271,48.841013,FV,South


In [34]:
stations_clean.info()

<class 'pandas.DataFrame'>
Index: 4600 entries, 0 to 6594
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   DS100         4600 non-null   str    
 1   NAME          4600 non-null   str    
 2   VERKEHR       4600 non-null   str    
 3   LAENGE        4600 non-null   float64
 4   BREITE        4600 non-null   float64
 5   station_type  4600 non-null   str    
 6   region        4600 non-null   str    
dtypes: float64(2), str(5)
memory usage: 287.5 KB


# Step 3.12: Export Clean Station Database

The cleaned station database is exported as a separate CSV file.

This file will serve as the input dataset for generating diverse training routes.

The route generation itself will be performed separately from this data cleaning process.

In [35]:
# Export cleaned station database

output_path = "../data/stations_for_route_generation.csv"

stations_clean.to_csv(
    output_path,
    index=False
)

output_path

'../data/stations_for_route_generation.csv'

In [36]:
#does the data exist?

In [37]:
import os

os.path.exists(output_path)

True

# Filtering stations for route generation (Correction)

The original station dataset contains all types of railway stations.
For the energy model, we want to avoid small urban stops and S-Bahn stations,
because they are not representative for potential long-distance railway routes.

The goal is to create a cleaner station database containing:
- Fernverkehr stations (FV)
- relevant regional railway stations (RV)

This filtered dataset will later be used to generate a diverse training set of routes.

In [38]:
# Check current station types

stations_filtered["VERKEHR"].value_counts()

VERKEHR
RV    4274
FV     326
Name: count, dtype: int64

In [39]:
# Remove obvious local/S-Bahn-like stations based on station names

s_bahn_keywords = [
    "S-Bahn",
    "Stadtmitte",
    "Rathaus",
    "Bürgerpark",
    "Zentrum",
    "Mitte",
    "West",
    "Ost",
    "Süd",
    "Nord"
]

pattern = "|".join(s_bahn_keywords)

stations_clean = stations_filtered[
    ~stations_filtered["NAME"].str.contains(
        pattern,
        case=False,
        na=False
    )
]


stations_clean.shape

(4212, 8)

In [40]:
removed = stations_filtered[
    stations_filtered["NAME"].str.contains(
        pattern,
        case=False,
        na=False
    )
]

removed.head(20)

,EVA_NR,DS100,NAME,VERKEHR,LAENGE,BREITE,Unnamed: 6,Unnamed: 7
2,8000404,KAW,Aachen West,RV,6.070715,50.780360,NaN,NaN
15,8000420,NADM,Adelsdorf(Mittelfr),RV,10.687041,49.474944,NaN,NaN
16,8000423,RADN,Adelsheim Nord,RV,9.392754,49.421490,NaN,NaN
17,8000424,TAD,Adelsheim Ost,RV,9.395127,49.403123,NaN,NaN
28,8000441,EAHL,Ahlen(Westf),RV,7.895488,51.761062,NaN,NaN
75,8000510,KADP,Alsdorf Poststraße,RV,6.197075,50.855749,NaN,NaN
88,8000504,NADW,Altdorf West,RV,11.341467,49.391146,NaN,NaN
135,8077274,FAZUN,Alzenau Nord,RV,9.051385,50.089443,NaN,NaN
138,8000558,FAZS,Alzey Süd,RV,8.118732,49.738134,NaN,NaN
139,8007474,FAZW,Alzey West,RV,8.107255,49.740978,NaN,NaN


In [41]:
stations_clean["VERKEHR"].value_counts()

VERKEHR
RV    3908
FV     304
Name: count, dtype: int64

In [42]:
stations_clean = stations_clean[
    stations_clean["VERKEHR"].isin(["FV", "RV"])
]

In [43]:
# Export cleaned station database

output_path = "../data/raw/stations_for_route_generation_cleaned.csv"

stations_clean.to_csv(
    output_path,
    index=False,
    encoding="utf-8"
)

print("Saved:", output_path)

Saved: ../data/raw/stations_for_route_generation_cleaned.csv


# Quality check of cleaned station database

After filtering out unsuitable stations, we check whether the remaining
station database is suitable for generating diverse railway routes.

The database should contain:
- different regions of Germany
- a mixture of FV and relevant RV stations
- enough stations for creating diverse routes
- valid geographic coordinates

This dataset will be used as the input for route generation.

In [44]:
stations_clean.shape

(4212, 8)

In [45]:
stations_clean["VERKEHR"].value_counts()

VERKEHR
RV    3908
FV     304
Name: count, dtype: int64

In [46]:
stations_clean["region"].value_counts()

KeyError: 'region'

In [47]:
stations_clean.columns

Index(['EVA_NR', 'DS100', 'NAME', 'VERKEHR', 'LAENGE', 'BREITE', 'Unnamed: 6',
       'Unnamed: 7'],
      dtype='str')

# Adding geographic regions

The route generation requires geographic diversity.
Therefore, each station receives a rough region classification
based on its coordinates.

These regions are only used for sampling diverse routes.
The actual railway characteristics will later come from the API.

In [48]:
def assign_region(row):
    lon = row["LAENGE"]
    lat = row["BREITE"]

    if lat >= 52.5:
        return "North"
    elif lon >= 12:
        return "East"
    elif lon <= 8:
        return "West"
    elif lat < 49:
        return "South"
    else:
        return "Central"


stations_clean["region"] = stations_clean.apply(assign_region, axis=1)

stations_clean["region"].value_counts()

region
Central    1405
South       806
West        788
East        670
North       543
Name: count, dtype: int64

In [49]:
stations_clean.shape

(4212, 9)

In [50]:
stations_clean["VERKEHR"].value_counts()

VERKEHR
RV    3908
FV     304
Name: count, dtype: int64

In [51]:
stations_clean["region"].value_counts()

region
Central    1405
South       806
West        788
East        670
North       543
Name: count, dtype: int64

In [53]:
stations_clean.shape

(4212, 9)

In [54]:
stations_clean["VERKEHR"].value_counts()

VERKEHR
RV    3908
FV     304
Name: count, dtype: int64

In [55]:
stations_clean["region"].value_counts()

region
Central    1405
South       806
West        788
East        670
North       543
Name: count, dtype: int64

In [56]:
# Export final cleaned station database for route generation

output_path = "../data/raw/stations_for_route_generation_final.csv"

stations_clean.to_csv(
    output_path,
    index=False,
    encoding="utf-8"
)

print("Saved:", output_path)

Saved: ../data/raw/stations_for_route_generation_final.csv


# Loading generated training routes . test of the trainingset!

The generated route dataset contains 100 start-end station combinations.
Before using these routes for API requests and energy modelling, we perform
quality checks regarding diversity, distance distribution and geographic coverage.

No modifications are made at this stage.

In [57]:
import pandas as pd

routes = pd.read_csv("../data/raw/training_routes_100b.csv")

routes.head()

,start_DS100,start_name,start_region,end_DS100,end_name,end_region,axis,distance_category,air_distance_km
0,TSG,Schwäbisch Gmünd,South,TDIH,Distelhausen,Central,Regional,short,88.97
1,BLS,Berlin Hbf,North,LBAS,Barleber See,Central,Regional,short,122.36
2,TPH,Pforzheim Hbf,South,HN,Northeim(Han),Central,N-S,medium,325.36
3,SSWD,St Wendel,West,USE,Seebach(Mühlhausen),Central,NE-SW,medium,304.31
4,UCB,Camburg(Saale),Central,TBLF,Blaufelden,Central,NE-SW,medium,231.00


In [58]:
routes.shape

(100, 9)

In [59]:
routes.shape

(100, 9)

In [60]:
routes.head()

,start_DS100,start_name,start_region,end_DS100,end_name,end_region,axis,distance_category,air_distance_km
0,TSG,Schwäbisch Gmünd,South,TDIH,Distelhausen,Central,Regional,short,88.97
1,BLS,Berlin Hbf,North,LBAS,Barleber See,Central,Regional,short,122.36
2,TPH,Pforzheim Hbf,South,HN,Northeim(Han),Central,N-S,medium,325.36
3,SSWD,St Wendel,West,USE,Seebach(Mühlhausen),Central,NE-SW,medium,304.31
4,UCB,Camburg(Saale),Central,TBLF,Blaufelden,Central,NE-SW,medium,231.00


# Check station diversity

We check whether the generated routes use diverse start and end stations.
A high number of unique stations prevents bias towards specific locations.


In [61]:
routes["start_name"].nunique(), routes["end_name"].nunique()

(88, 98)

In [62]:
routes["start_name"].value_counts().head(10)

start_name
Steinach(b Rothenburg ob der Tauber)    2
Büchen                                  2
Kaufbeuren                              2
Augsburg Hbf                            2
Immendingen                             2
Delmenhorst                             2
Peine                                   2
Eberswalde Hbf                          2
Hornberg(Schwarzw)                      2
Sierksdorf                              2
Name: count, dtype: int64

In [63]:
routes["end_name"].value_counts().head(10)

end_name
Oppendorf Bahnhof      2
Sierksdorf             2
Distelhausen           1
Barleber See           1
Northeim(Han)          1
Seebach(Mühlhausen)    1
Blaufelden             1
Husum                  1
Cham(Oberpf)           1
Stolberg(Rheinl)Hbf    1
Name: count, dtype: int64

# Distance distribution

The training set should contain a balanced mixture of short, medium and long
routes to allow the energy model to learn different distance characteristics.

In [64]:
routes["distance_category"].value_counts()

distance_category
medium    45
long      30
short     25
Name: count, dtype: int64

In [65]:
routes["air_distance_km"].describe()

count    100.000000
mean     367.384700
std      188.472895
min       72.490000
25%      201.127500
50%      354.620000
75%      522.252500
max      734.720000
Name: air_distance_km, dtype: float64

# Geographic axis distribution

The dataset should cover different geographical directions across Germany.

In [66]:
routes["axis"].value_counts()

axis
N-S         34
NE-SW       31
Regional    15
NW-SE       14
E-W          6
Name: count, dtype: int64

# Test zeigt: sehr passendes Testset aber: welche Routen sind "Regional"?`

routes[routes["axis"] == "Regional"]

# Correcting geographical route axes

Some routes were incorrectly labelled as "Regional".
The route itself is still useful for the energy model.

We therefore recalculate the geographical axis based on
the coordinates of start and end stations.

In [68]:
stations_clean.head()

,EVA_NR,DS100,NAME,VERKEHR,LAENGE,BREITE,Unnamed: 6,Unnamed: 7,region
0,8000001,KA,Aachen Hbf,FV,6.091499,50.767800,NaN,NaN,West
1,8070704,KASZ,Aachen Schanz,RV,6.073840,50.769862,NaN,NaN,West
3,8000406,KAREP,Aachen-Rothe Erde,RV,6.116475,50.770202,NaN,NaN,West
4,8000002,TA,Aalen,FV,10.096271,48.841013,NaN,NaN,South
7,8000412,RAH,Achern,RV,8.065324,48.633990,NaN,NaN,South


In [69]:
stations_clean.columns

Index(['EVA_NR', 'DS100', 'NAME', 'VERKEHR', 'LAENGE', 'BREITE', 'Unnamed: 6',
       'Unnamed: 7', 'region'],
      dtype='str')

# Adding station coordinates to training routes

The route dataset only contains DS100 station codes.
We merge it with the cleaned station database to add:

- start coordinates
- end coordinates
- station information

This allows us to recalculate the geographical route axis.

In [70]:
# Create lookup table for stations

stations_lookup = stations_clean[
    ["DS100", "NAME", "LAENGE", "BREITE", "region"]
].copy()

stations_lookup.head()

,DS100,NAME,LAENGE,BREITE,region
0,KA,Aachen Hbf,6.091499,50.767800,West
1,KASZ,Aachen Schanz,6.073840,50.769862,West
3,KAREP,Aachen-Rothe Erde,6.116475,50.770202,West
4,TA,Aalen,10.096271,48.841013,South
7,RAH,Achern,8.065324,48.633990,South


In [71]:
# Add start station information

routes_geo = routes.merge(
    stations_lookup,
    left_on="start_DS100",
    right_on="DS100",
    how="left"
)

routes_geo = routes_geo.rename(columns={
    "NAME": "start_station_check",
    "LAENGE": "start_longitude",
    "BREITE": "start_latitude",
    "region": "start_region_check"
})

routes_geo.head()

,start_DS100,start_name,start_region,end_DS100,end_name,end_region,axis,distance_category,air_distance_km,DS100,start_station_check,start_longitude,start_latitude,start_region_check
0,TSG,Schwäbisch Gmünd,South,TDIH,Distelhausen,Central,Regional,short,88.97,TSG,Schwäbisch Gmünd,9.787795,48.801011,South
1,BLS,Berlin Hbf,North,LBAS,Barleber See,Central,Regional,short,122.36,BLS,Berlin Hbf,13.369545,52.525592,North
2,TPH,Pforzheim Hbf,South,HN,Northeim(Han),Central,N-S,medium,325.36,TPH,Pforzheim Hbf,8.703099,48.894152,South
3,SSWD,St Wendel,West,USE,Seebach(Mühlhausen),Central,NE-SW,medium,304.31,SSWD,St Wendel,7.165293,49.466952,West
4,UCB,Camburg(Saale),Central,TBLF,Blaufelden,Central,NE-SW,medium,231.00,UCB,Camburg(Saale),11.704680,51.050857,Central


In [72]:
# Add end station information

routes_geo = routes_geo.merge(
    stations_lookup,
    left_on="end_DS100",
    right_on="DS100",
    how="left",
    suffixes=("", "_end")
)

routes_geo = routes_geo.rename(columns={
    "NAME_end": "end_station_check",
    "LAENGE_end": "end_longitude",
    "BREITE_end": "end_latitude",
    "region_end": "end_region_check"
})

routes_geo.head()

,start_DS100,start_name,start_region,end_DS100,end_name,end_region,axis,distance_category,air_distance_km,DS100,start_station_check,start_longitude,start_latitude,start_region_check,DS100_end,NAME,LAENGE,BREITE,region
0,TSG,Schwäbisch Gmünd,South,TDIH,Distelhausen,Central,Regional,short,88.97,TSG,Schwäbisch Gmünd,9.787795,48.801011,South,TDIH,Distelhausen,9.683508,49.598233,Central
1,BLS,Berlin Hbf,North,LBAS,Barleber See,Central,Regional,short,122.36,BLS,Berlin Hbf,13.369545,52.525592,North,LBAS,Barleber See,11.640852,52.214328,Central
2,TPH,Pforzheim Hbf,South,HN,Northeim(Han),Central,N-S,medium,325.36,TPH,Pforzheim Hbf,8.703099,48.894152,South,HN,Northeim(Han),9.986618,51.703139,Central
3,SSWD,St Wendel,West,USE,Seebach(Mühlhausen),Central,NE-SW,medium,304.31,SSWD,St Wendel,7.165293,49.466952,West,USE,Seebach(Mühlhausen),10.521193,51.170012,Central
4,UCB,Camburg(Saale),Central,TBLF,Blaufelden,Central,NE-SW,medium,231.00,UCB,Camburg(Saale),11.704680,51.050857,Central,TBLF,Blaufelden,9.967680,49.296236,Central


In [73]:
routes_geo.shape

(100, 19)

In [74]:
routes_geo[
    [
        "start_name",
        "start_station_check",
        "end_name",
        "end_station_check"
    ]
].head(10)

KeyError: "['end_station_check'] not in index"

In [75]:
routes_geo.columns

Index(['start_DS100', 'start_name', 'start_region', 'end_DS100', 'end_name',
       'end_region', 'axis', 'distance_category', 'air_distance_km', 'DS100',
       'start_station_check', 'start_longitude', 'start_latitude',
       'start_region_check', 'DS100_end', 'NAME', 'LAENGE', 'BREITE',
       'region'],
      dtype='str')

# Renaming end station information

The end station information was merged successfully.
We rename the columns to make the dataset easier to understand.

In [76]:
routes_geo = routes_geo.rename(columns={
    "NAME": "end_station_check",
    "LAENGE": "end_longitude",
    "BREITE": "end_latitude",
    "region": "end_region_check"
})

routes_geo.head()

,start_DS100,start_name,start_region,end_DS100,end_name,end_region,axis,distance_category,air_distance_km,DS100,start_station_check,start_longitude,start_latitude,start_region_check,DS100_end,end_station_check,end_longitude,end_latitude,end_region_check
0,TSG,Schwäbisch Gmünd,South,TDIH,Distelhausen,Central,Regional,short,88.97,TSG,Schwäbisch Gmünd,9.787795,48.801011,South,TDIH,Distelhausen,9.683508,49.598233,Central
1,BLS,Berlin Hbf,North,LBAS,Barleber See,Central,Regional,short,122.36,BLS,Berlin Hbf,13.369545,52.525592,North,LBAS,Barleber See,11.640852,52.214328,Central
2,TPH,Pforzheim Hbf,South,HN,Northeim(Han),Central,N-S,medium,325.36,TPH,Pforzheim Hbf,8.703099,48.894152,South,HN,Northeim(Han),9.986618,51.703139,Central
3,SSWD,St Wendel,West,USE,Seebach(Mühlhausen),Central,NE-SW,medium,304.31,SSWD,St Wendel,7.165293,49.466952,West,USE,Seebach(Mühlhausen),10.521193,51.170012,Central
4,UCB,Camburg(Saale),Central,TBLF,Blaufelden,Central,NE-SW,medium,231.00,UCB,Camburg(Saale),11.704680,51.050857,Central,TBLF,Blaufelden,9.967680,49.296236,Central


In [77]:
routes_geo[
    [
        "start_name",
        "start_station_check",
        "end_name",
        "end_station_check"
    ]
].head(10)

,start_name,start_station_check,end_name,end_station_check
0,Schwäbisch Gmünd,Schwäbisch Gmünd,Distelhausen,Distelhausen
1,Berlin Hbf,Berlin Hbf,Barleber See,Barleber See
2,Pforzheim Hbf,Pforzheim Hbf,Northeim(Han),Northeim(Han)
3,St Wendel,St Wendel,Seebach(Mühlhausen),Seebach(Mühlhausen)
4,Camburg(Saale),Camburg(Saale),Blaufelden,Blaufelden
5,Soest,Soest,Oppendorf Bahnhof,Oppendorf Bahnhof
6,Villingen(Schwarzw),Villingen(Schwarzw),Husum,Husum
7,Steinach(b Rothenburg ob der Tauber),Steinach(b Rothenburg ob der Tauber),Cham(Oberpf),Cham(Oberpf)
8,Bad Oeynhausen,Bad Oeynhausen,Stolberg(Rheinl)Hbf,Stolberg(Rheinl)Hbf
9,Büchen,Büchen,Umrathshausen Bf,Umrathshausen Bf


# Recalculating geographical route axes

The previous dataset contained some routes labelled as "Regional".
This category does not describe a geographical direction.

We calculate the axis again based on the longitude and latitude difference
between start and end stations.

In [78]:
import numpy as np

def calculate_axis(row):
    lon_diff = row["end_longitude"] - row["start_longitude"]
    lat_diff = row["end_latitude"] - row["start_latitude"]

    # absolute differences
    lon = abs(lon_diff)
    lat = abs(lat_diff)

    # mostly east-west
    if lon > 2 * lat:
        return "E-W"

    # mostly north-south
    elif lat > 2 * lon:
        return "N-S"

    # diagonal directions
    else:
        if lon_diff > 0 and lat_diff > 0:
            return "NE-SW"
        elif lon_diff < 0 and lat_diff > 0:
            return "NW-SE"
        elif lon_diff > 0 and lat_diff < 0:
            return "NE-SW"
        else:
            return "NW-SE"


routes_geo["axis_corrected"] = routes_geo.apply(
    calculate_axis,
    axis=1
)

routes_geo["axis_corrected"].value_counts()

axis_corrected
N-S      33
NW-SE    25
NE-SW    23
E-W      19
Name: count, dtype: int64

# Checking geographical coverage

We check whether the training routes cover different regions of Germany.
A diverse spatial distribution helps the model learn different infrastructure
and topographical characteristics.routes_geo["start_region"].value_counts()

In [79]:
routes_geo["start_region"].value_counts()

start_region
North      30
South      26
Central    25
East       12
West        7
Name: count, dtype: int64

In [80]:
routes_geo["end_region"].value_counts()

end_region
Central    31
North      25
West       18
East       14
South      12
Name: count, dtype: int64

# Saving merged route dataset

The dataset now contains:
- route information
- station coordinates
- regional information
- corrected geographical axis

This version is saved as a backup before further processing.

In [81]:
routes_geo.to_csv(
    "../data/raw/training_routes_100_with_coordinates.csv",
    index=False
)

print("File saved successfully!")

File saved successfully!


In [82]:
import os
os.getcwd()

'/Users/Helena/00_Workspace/Programming/Projects/night-train-target-network/Target_Network2026/backend/models/energy/notebooks'

In [83]:
import os
os.getcwd()

'/Users/Helena/00_Workspace/Programming/Projects/night-train-target-network/Target_Network2026/backend/models/energy/notebooks'